# 🔥 Advanced · DreamerV3 (world-model RL)

> 🔥 **Advanced / heavy lab.** Learn a latent world model from pixels and train an agent inside it ('learning in imagination').
>
> **Runtime → Change runtime type → T4 GPU required.** Est. **1–3 h for visible progress** including downloads. Built on the official **[danijar/dreamerv3](https://github.com/danijar/dreamerv3)** and authored to its documented recipe — **not pre-executed here** (needs a GPU + large downloads). If a step fails, see *Troubleshooting* at the bottom; pin versions as noted.

Maps to lesson **D8 (world models)** — the real, scalable version of the toy world-model + planning lab.

## Compute · storage · time

| Resource | Demo (this notebook, free Colab T4) | Full / production run |
|---|---|---|
| **GPU** | 1× T4 / 24 GB — Crafter, partial budget | 1× A100 40 GB |
| **Storage** | replay buffer + logs ~ few GB | replay buffer 10–50 GB (pixel envs) + logs |
| **Time** | ~ hours for 1e5 steps | ~ 1–3 days for 1e6–5e6 steps |

**Full pipeline (scale-up):** `python dreamerv3/main.py --configs <task> --run.steps 5e6 --logdir …`.

> Rough estimates — real numbers depend on hardware, data size and library versions.

In [ ]:
!nvidia-smi

## 1 · Install
DreamerV3 is JAX-based. Install the package (or clone the repo for the latest configs).

In [ ]:
%cd /content
!git clone https://github.com/danijar/dreamerv3.git
%cd dreamerv3
!pip install -q -r requirements.txt
!pip install -q "jax[cuda12]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

## 2 · Train on a task (Crafter — a fast, open-ended benchmark)

In [ ]:
!python dreamerv3/main.py \
  --logdir ./logdir/crafter_run \
  --configs crafter \
  --run.steps 1e5

## 3 · Inspect what the world model "imagines"
Dreamer logs open-loop video predictions (true vs. imagined frames) and metrics to `logdir`. View them with TensorBoard:

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ./logdir

## Evaluate — mean episode return
The metric is **mean evaluation-episode return** vs environment steps (DreamerV3 logs `eval/return` to the logdir / TensorBoard). Average the return over several eval episodes with the trained agent and compare to the task's reported score.

## Save & persist your results
This pipeline writes its checkpoints, metrics/logs and outputs to the run/output directory shown above (e.g. `output/`, `outputs/`, `logdir/`, `experiments/`, or the trainer's `output_dir`). **Colab sessions are ephemeral** — to keep them, either mount Drive and copy the folder (`from google.colab import drive; drive.mount('/content/drive')`) or zip + download it (`import shutil; shutil.make_archive('run','zip','OUTPUT_DIR')` then `from google.colab import files; files.download('run.zip')`). The 🤗 Trainer labs also support `trainer.push_to_hub()`. To publish any output folder as a **model repo** (then group repos into a **Collection** on your profile): `from huggingface_hub import HfApi; HfApi().upload_folder(folder_path="OUTPUT_DIR", repo_id="<you>/<lab>")`.

## How this links to tracks A–D
World models are the engine of **AG · Agents & RL** model-based agents.

## Troubleshooting & next steps
- **JAX/GPU**: match the `jax[cuda12]` wheel to the runtime's CUDA; if it errors, use the version pinned in the repo's README.
- **Other tasks**: `--configs dmc_vision` (DeepMind Control from pixels), `atari`, etc.
- Flags shift between releases — check `python dreamerv3/main.py --help`.
- Lighter, fully-runnable alternative: the self-contained **world model + planning** lab in `notebooks/training/`.